In [2]:
!pip install -q torch==2.1.1 datasets==2.17.1 scipy==1.12.0 hf_transfer==0.1.5 huggingface-hub==0.25.0 wandb==0.16.3 wheel==0.44.0 transformers==4.44.2 accelerate==0.34.2 peft==0.12.0 "trl<0.9.0" bitsandbytes==0.43.3 deepspeed==0.15.1 einops==0.8.0 sentencepiece==0.2.0 nltk==3.9.1 xformers==0.0.23 unsloth==2024.8 flash-attn==2.6.3 --no-build-isolation


[notice] A new release of pip is available: 23.3.1 -> 24.2
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
import os

os.environ['HF_HOME'] = '/workspace/persistent'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = 'True'

from huggingface_hub import login
login(token="HF_TOKEN_PLACEHOLDER")

hf_home = os.getenv('HF_HOME')
print(hf_home)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /workspace/persistent/token
Login successful
/workspace/persistent


In [5]:
import yaml

# Load the existing YAML config file
config_file = "fsdp_config.yaml"

# Open and read the YAML file
with open(config_file, 'r') as file:
    config = yaml.safe_load(file)

# Modify the 'num_processes' value
config['num_processes'] = 1  # Change this to the desired number

# Write the updated config back to the YAML file
with open(config_file, 'w') as file:
    yaml.safe_dump(config, file)

print(f"Updated num_processes to {config['num_processes']} in {config_file}")

Updated num_processes to 1 in fsdp_config.yaml


In [ ]:
!python train.py \
--lora_r 16 \
--lora_alpha 32 \
--lora_dropout 0.0 \
--lora_target_modules "q_proj,k_proj,v_proj,o_proj" \
--use_4bit_quantization True \
--model_name_or_path "meta-llama/Meta-Llama-3.1-8B-Instruct" \
--max_seq_len 1024 \
--learning_rate 1e-4 \
--lr_scheduler_type "cosine" \
--warmup_ratio 0.00 \
--max_grad_norm 1.0 \
--per_device_train_batch_size 1 \
--gradient_accumulation_steps 1 \
--num_train_epochs 15 \
--dataset_text_field "text" \
--output_dir "llama-sft-lora-fsdp" \
--use_peft_lora True \
--seed 100 \
--seed 100 \
--seed 100 \
--dataset_name "smangrul/ultrachat-10k-chatml" \
--chat_template_format "chatml" \
--add_special_tokens False \
--append_concat_token False \
--splits "train" \
--logging_steps 1 \
--log_level "info" \
--logging_strategy "steps" \
--evaluation_strategy "no" \
--save_strategy "no" \
--hub_private_repo True \
--hub_strategy "every_save" \
--bf16 True \
--packing False \
--per_device_eval_batch_size 1 \
--gradient_checkpointing True \
--use_reentrant False \
--use_flash_attn True \
--use_unsloth True

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Your GPU supports bfloat16, you can accelerate training with the argument --bf16
==((====))==  Unsloth 2024.8: Fast Llama patching. Transformers = 4.44.2.
   \\   /|    GPU: NVIDIA A40. Max memory: 44.339 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.1.1+cu121. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.23. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
model.safetensors: 100%|███████████████████▉| 5.70G/5.70G [00:14<00: